# 🏛️ Qwen3.5-0.8B Vision — Historical Spanish OCR LoRA Fine-tuning
> Built with [Unsloth](https://unsloth.ai) · Tracks CER on Validation & Test sets

**Notebook outline:**
1. Install dependencies
2. Configuration (edit here)
3. Authentication (HF + optional W&B)
4. Load model + LoRA adapter
5. Text normalisation (derived from train/test format analysis)
6. Load dataset (train / validation / test splits)
7. Image scaling to match inference
8. Optional data augmentation
9. Format dataset for vision fine-tuning
10. Pre-training baseline (CER snapshot)
11. Live dashboard callback (loss + val CER)
12. Train
13. Post-training validation CER
14. Test set evaluation (run once at the very end)
15. Save adapter & push to HuggingFace Hub
16. Expert notes & recommendations


### 1 · Installation

In [1]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
# Extra: dashboard, CER metric, augmentation, HF upload
!uv pip install -qqq matplotlib jiwer albumentations huggingface_hub wandb
!uv pip install flash-attn --no-build-isolation

In [2]:
import flash_attn
print(flash_attn.__version__)

2.8.3


### 2 · Configuration — **Edit these cells**

In [ ]:
from kaggle_secrets import UserSecretsClient
secret_label = "HF_TOKEN"
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
wandb_token = UserSecretsClient().get_secret("WANDB")

In [ ]:
# ════════════════════════════════════════════════════════════
#  USER CONFIG  —  fill in before running
# ════════════════════════════════════════════════════════════

# ── Dataset ──────────────────────────────────────────────────
HF_DATASET_NAME  = "Ak137/Handwritten_Historical_Spanish"   # @param {type:"string"}
TRAIN_SPLIT      = "train"                         # @param {type:"string"}
VAL_SPLIT        = "validation"                    # @param {type:"string"}
TEST_SPLIT       = "test"                          # @param {type:"string"}

# ── HuggingFace Hub ───────────────────────────────────────────
HF_USERNAME      = "Ak137"                 # @param {type:"string"}
HF_TOKEN         = hf_token                              # @param {type:"string"}
ADAPTER_REPO_ID  = f"{HF_USERNAME}/qwen3.5-0.8B-handwritten-spanish-ocr-lora"

# ── Weights & Biases (optional) ───────────────────────────────
USE_WANDB        = True                           # @param {type:"boolean"}
WANDB_TOKEN      =  wandb_token # @param {type:"string"}
WANDB_PROJECT    = "Qwen3.5-0.8b-spanish-OCR"              # @param {type:"string"}

# ── Model ─────────────────────────────────────────────────────
MODEL_NAME       = "unsloth/Qwen3.5-0.8B"
LOAD_IN_4BIT     = False
IMAGE_MAX_DIM    = 2048

# ── Instruction ───────────────────────────────────────────────
INSTRUCTION = (
    "Transcribe the historical Spanish text in the provided image exactly as written."
    "RULES:"
    "1. Preserve all original archaic spellings and abbreviations and line breaks. Do not modernize the text."
    "2. Output STRICTLY in plain text."
    "3. ABSOLUTELY NO HTML, XML, Markdown, or structural tags (no <div>, <p>, etc.)."
    "4. NO conversational filler, headers, or commentary."
    "### Transcription:"
)

# ════════════════════════════════════════════════════════════
#  TRAINING HYPER-PARAMETERS
#  Calibrated for ~2000 train / 500 val / 500 test
# ════════════════════════════════════════════════════════════

# LoRA
LORA_R                    = 16    # r=16 is appropriate for 2k samples on a 0.8B model
LORA_ALPHA                = 32    # 2*r
LORA_DROPOUT              = 0.0  # slightly higher than default; helps with 2k samples

# Training schedule
NUM_EPOCHS                = 5     # ~1250 gradient steps; enough to converge without memorising
MAX_STEPS                 = -1    # override epochs only for quick smoke-tests

PER_DEVICE_BATCH_SIZE     = 24
GRADIENT_ACCUMULATION     = 1     # effective batch = 8

LEARNING_RATE             = 1e-4
LR_SCHEDULER              = "cosine"
# WARMUP_RATIO              = 0.05
WARMUP_STEPS              = 33
WEIGHT_DECAY              = 0.01
MAX_GRAD_NORM             = 1.0
MAX_SEQ_LENGTH            = 2048

SAVE_STEPS                = 50
LOGGING_STEPS             = 5
OUTPUT_DIR                = "outputs"

# ── Evaluation ────────────────────────────────────────────────
# Val CER is computed on a random subset every EVAL_EVERY_N_STEPS to stay fast.
# Full val + test CER run at the very end.
# EVAL_EVERY_N_STEPS        = 87   # adjust to ~1 eval per epoch
VAL_EVAL_SUBSET           = 50    # samples used for mid-training val CER (speed vs accuracy)
EVAL_MAX_NEW_TOKENS       = 2048   # generation budget during eval
EVAL_BATCH_SIZE           = 48

# ── Data augmentation (optional but recommended) ──────────────
USE_AUGMENTATION          = True  # light geometric + photometric augmentation
AUGMENTATION_PROB         = 0.5   # probability of applying aug to any single sample

### 3 · Authentication

In [5]:
from huggingface_hub import login as hf_login

if HF_TOKEN:
    hf_login(token=HF_TOKEN, add_to_git_credential=False)
    print("✅ Logged in to HuggingFace Hub")
else:
    print("⚠️  No HF_TOKEN — private datasets / Hub push will be skipped.")

if USE_WANDB:
    import wandb
    wandb.login(key= WANDB_TOKEN)
    print("✅ Logged in to Weights & Biases")

✅ Logged in to HuggingFace Hub


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /teamspace/studios/this_studio/.netrc
wandb: Currently logged in as: adhams-ka7 (adhams-ka7-ain-shams-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ Logged in to Weights & Biases


### 4 · Load Model & LoRA Adapter

In [9]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit               = LOAD_IN_4BIT,
    use_gradient_checkpointing = "unsloth",
    attn_implementation        = "flash_attention_2",
    # dtype = torch.float32,
)
tokenizer.padding_side = "left"
tokenizer.image_processor.padding_side = "left"
if hasattr(tokenizer, 'tokenizer'):
    tokenizer.tokenizer.padding_side = "left"   # ← the actual inner tokenizer

tokenizer.image_processor.max_pixels = IMAGE_MAX_DIM * IMAGE_MAX_DIM   # 4,194,304
tokenizer.image_processor.min_pixels = 256 * 256
print(f"✅ Loaded {MODEL_NAME}  |  dtype={model.dtype}")

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,   # critical for manuscript textures
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r            = LORA_R,
    lora_alpha   = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias         = "none",
    random_state = 3407,
    use_rslora   = False,
    loftq_config = None,
)
model.print_trainable_parameters()

==((====))==  Unsloth 2026.3.15: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

✅ Loaded unsloth/Qwen3.5-0.8B  |  dtype=torch.bfloat16
Unsloth: Making `model.base_model.model.model.visual` require gradients
trainable params: 13,181,952 || all params: 866,167,872 || trainable%: 1.5219


### 5 · Text Normalisation

Rules derived by comparing train labels (historical Spanish print/manuscript)
against the test evaluation notes:

| Rule | Rationale | Example |
|---|---|---|
| `ſ` → `s` | Long-s not in test format | `ſuelto` → `suelto` |
| `ç`/`Ç` → `z`/`Z` | Explicitly stated in test notes | `çapatos` → `zapatos` |
| Nasal tilde abbreviations | `ã õ ẽ ũ ĩ` = vowel + contracted n | `razõ` → `razon` |
| `đ` → `de` | D-with-stroke = preposition *de* | `alma đ dexar` → `alma de dexar` |
| Strip stress accents *(keep ñ)* | Test eval ignores accents except ñ | `fè` → `fe`, `cañón` → `cañon` |
| Collapse multiple spaces | Clean label noise | `buſcae  quiero` → `buscae quiero` |

u/v and f/s substitutions are **intentionally not normalised** — the test notes say
"check against dictionary" meaning those variants are preserved verbatim in evaluation.


In [10]:
import unicodedata
import re

_NASAL_MAP = {
    'ã': 'an', 'Ã': 'An',
    'õ': 'on', 'Õ': 'On',
    'ẽ': 'en', 'Ẽ': 'En',
    'ũ': 'un', 'Ũ': 'Un',
    'ĩ': 'in', 'Ĩ': 'In',
}

def normalize_text(text: str) -> str:
    """Apply all normalisation rules derived from train/test format analysis."""
    # Rule 1 — long-s
    text = text.replace('ſ', 's')
    # Rule 2 — cedilla
    text = text.replace('ç', 'z').replace('Ç', 'Z')
    # Rule 3 — nasal tilde abbreviations (before accent stripping)
    for ch, expansion in _NASAL_MAP.items():
        text = text.replace(ch, expansion)
    # Rule 4 — d-with-stroke
    text = text.replace('đ', 'de').replace('Đ', 'De')
    # Rule 5 — strip stress accents, keep ñ
    nfd = unicodedata.normalize('NFD', text)
    out, i = [], 0
    while i < len(nfd):
        c = nfd[i]
        if (c in ('n', 'N')
                and i + 1 < len(nfd)
                and nfd[i + 1] == '\u0303'):
            out.append(c); out.append('\u0303'); i += 2
        elif unicodedata.category(c) == 'Mn':
            i += 1
        else:
            out.append(c); i += 1
    text = unicodedata.normalize('NFC', ''.join(out))
    # Rule 6 — collapse spaces, preserve newlines
    text = re.sub(r'[ \t]+', ' ', text).strip()
    return text

# ── Sanity checks ─────────────────────────────────────────────────────────
_tests = [
    ("ſuelto ſoegada",       "suelto soegada"),
    ("razõ clara",           "razon clara"),
    ("el alma đ dexarme",    "el alma de dexarme"),
    ("çapatos",              "zapatos"),
    ("fè del amor",          "fe del amor"),
    ("cañón",                "cañon"),
    ("buſcae  quiero",       "buscae quiero"),
]
ok = all(normalize_text(r) == e for r, e in _tests)
for raw, expected in _tests:
    result = normalize_text(raw)
    print(f"{'✅' if result==expected else '❌'}  '{raw}' → '{result}'")
print("\n✅ All tests passed!" if ok else "\n❌ Check failed!")

✅  'ſuelto ſoegada' → 'suelto soegada'
✅  'razõ clara' → 'razon clara'
✅  'el alma đ dexarme' → 'el alma de dexarme'
✅  'çapatos' → 'zapatos'
✅  'fè del amor' → 'fe del amor'
✅  'cañón' → 'cañon'
✅  'buſcae  quiero' → 'buscae quiero'

✅ All tests passed!


### 6 · Load Dataset (train / validation / test)

In [11]:
from datasets import load_dataset

print("Loading splits …")
ds_train = load_dataset(HF_DATASET_NAME, split=TRAIN_SPLIT)
ds_val   = load_dataset(HF_DATASET_NAME, split=VAL_SPLIT)
ds_test  = load_dataset(HF_DATASET_NAME, split=TEST_SPLIT)

print(f"  Train      : {len(ds_train):>5,} samples")
print(f"  Validation : {len(ds_val):>5,} samples")
print(f"  Test       : {len(ds_test):>5,} samples")
print(f"  Keys       : {ds_train.column_names}")

Loading splits …


Generating train split:   0%|          | 0/536 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/30 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/30 [00:00<?, ? examples/s]

  Train      :   536 samples
  Validation :    30 samples
  Test       :    30 samples
  Keys       : ['image', 'text', 'doc_id']


### 7 · Image Scaling

In [12]:
from PIL import Image as PILImage

def resize_image(img: PILImage.Image, max_dim: int = IMAGE_MAX_DIM) -> PILImage.Image:
    """Proportionally downscale so max(width, height) <= max_dim. LANCZOS quality."""
    w, h = img.size
    if max(w, h) <= max_dim:
        return img
    scale = max_dim / max(w, h)
    return img.resize((int(w * scale), int(h * scale)), PILImage.LANCZOS)

# Check
_img = ds_train[0]["image"]
print(f"Original  : {_img.size}")
print(f"Resized   : {resize_image(_img).size}  (max_dim={IMAGE_MAX_DIM})")

Original  : (1415, 2048)
Resized   : (1415, 2048)  (max_dim=2048)


### 8 · Data Augmentation (optional)

Light geometric + photometric augmentation tailored for historical manuscripts.
Applied **only to training images**, never to val/test.
Augmentation happens **after** resize so the final resolution stays consistent.


In [13]:
import numpy as np
import random

if USE_AUGMENTATION:
    import albumentations as A
    import cv2

    _aug_pipeline = A.Compose([
        # Geometric — simulate page warp / scan skew
        A.Rotate(limit=2, border_mode=cv2.BORDER_REPLICATE, p=0.5),
        A.Perspective(scale=(0.01, 0.03), p=0.3),
        A.ElasticTransform(alpha=1, sigma=10, p=0.2),
        # Photometric — simulate ink fading, yellowing, uneven lighting
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
        A.GaussNoise(std_range=(0.01, 0.05), p=0.3),   # replace var_limit with std_range
        A.ImageCompression(quality_range=(70, 95), p=0.2),  # scan compression artefacts
    ])

    def augment_image(img: PILImage.Image) -> PILImage.Image:
        """Apply random augmentation (returns PIL Image)."""
        if random.random() > AUGMENTATION_PROB:
            return img
        arr = np.array(img.convert("RGB"))
        aug = _aug_pipeline(image=arr)["image"]
        return PILImage.fromarray(aug)

    print("✅ Augmentation pipeline ready (applied to training images only)")
else:
    def augment_image(img: PILImage.Image) -> PILImage.Image:
        return img
    print("ℹ️  Augmentation disabled — set USE_AUGMENTATION=True to enable")

✅ Augmentation pipeline ready (applied to training images only)


### 9 · Format Dataset for LoRA Training

In [14]:
def convert_to_conversation(sample: dict, augment: bool = False) -> dict:
    """
    Convert {image, text} → Unsloth VLM conversation format.
    augment=True applies photometric/geometric augmentation (training only).
    """
    img  = resize_image(sample["image"])
    if augment:
        img = augment_image(img)
    text = normalize_text(sample["text"])

    return {
        "messages": [
            {"role": "user", "content": [
                {"type": "text",  "text":  INSTRUCTION},
                {"type": "image", "image": img},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": text},
            ]},
        ]
    }

print("Converting training set …")
converted_train = [convert_to_conversation(s, augment=USE_AUGMENTATION)
                   for s in ds_train]
print(f"✅ {len(converted_train):,} training samples ready")

# Val / test are NOT converted to conversation format for evaluation;
# we'll call the model directly on raw images and compare to normalize_text(gt).
# We still pre-compute normalised reference strings for fast CER calculation.
val_references  = [normalize_text(s["text"]) for s in ds_val]
test_references = [normalize_text(s["text"]) for s in ds_test]
print(f"✅ {len(val_references):,} val references  |  {len(test_references):,} test references")

Converting training set …
✅ 536 training samples ready
✅ 30 val references  |  30 test references


### 10 · Pre-Training Baseline CER

In [15]:
from jiwer import cer as compute_cer
from tqdm.auto import tqdm
import random as _random
import re

def truncate_repetition_loop(text: str, 
                              ngram_size: int = 8,
                              repeat_threshold: int = 4) -> str:
    """
    Detect and truncate degenerate repetition loops in OCR output.
    
    Works by scanning for any n-gram that repeats >= repeat_threshold
    times consecutively. Truncates at the first such occurrence.
    
    This is O(n) once on CPU after generation, not O(n^2) during generation.
    
    ngram_size=8: catches "!!!!!!!!" style loops and short phrase loops
    repeat_threshold=4: 4 consecutive repeats = definitely a loop
    """
    if not text:
        return text
    
    chars = list(text)
    n = len(chars)
    
    for ng in range(1, ngram_size + 1):
        for i in range(n - ng * repeat_threshold):
            ngram = text[i : i + ng]
            # Check if it repeats repeat_threshold times consecutively
            repeated = ngram * repeat_threshold
            if text[i : i + ng * repeat_threshold] == repeated:
                # Truncate at the start of the repetition, strip trailing whitespace
                return text[:i].rstrip()
    
    return text
def evaluate_cer(dataset_split, references: list,
                 model, tokenizer, n_samples: int = None,
                 desc: str = "Eval") -> float:
    """
    Batched CER evaluation with tqdm progress bar.
    Left-padding + batched generation maximises GPU utilisation.
    n_samples: random subset size (None = full split).
    """
    indices = list(range(len(dataset_split)))
    if n_samples and n_samples < len(indices):
        indices = _random.sample(indices, n_samples)

    hypotheses = []

    for batch_start in tqdm(range(0, len(indices), EVAL_BATCH_SIZE),
                             desc=desc, unit="batch"):
        batch_indices = indices[batch_start : batch_start + EVAL_BATCH_SIZE]

        # ── build inputs for the whole batch ──────────────────────────────
        images, texts = [], []
        for idx in batch_indices:
            img = resize_image(dataset_split[idx]["image"])
            messages = [{"role": "user", "content": [
                {"type": "text",  "text": INSTRUCTION},
                {"type": "image"},
            ]}]
            text = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True
            )
            images.append(img)
            texts.append(text)

        inputs = tokenizer(
            images,
            texts,
            padding       = True,
            add_special_tokens = False,
            return_tensors = "pt",
        ).to("cuda")

        prompt_len = inputs["input_ids"].shape[1]

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens = EVAL_MAX_NEW_TOKENS,
                use_cache      = True,
                # temperature    = 1.0,
                do_sample      = False,
                pad_token_id   = tokenizer.pad_token_id,
                # repetition_penalty = 1.3,    # ← penalises repeating the same token
                # no_repeat_ngram_size = 4,    # ← hard-blocks any 4-gram from repeating exactly
            )

        # Decode only the generated tokens (strip padded prompt)
        for seq in output_ids:
            hyp = tokenizer.decode(seq[prompt_len:], skip_special_tokens=True)
            hypotheses.append(truncate_repetition_loop(hyp))

    refs = [references[i] for i in indices]
    cer_score = compute_cer(refs, hypotheses)
    print(f"  [{desc}]  n={len(indices)}  CER = {cer_score:.4f}  ({cer_score*100:.2f} %)")
    # print("🔎 Sample Predictions:\n")
    # for (pred, ref) in zip(hypotheses, refs):
    #     print(f"GT   : {ref[:50]}")
    #     print(f"PRED : {pred[:50]}")
    #     print()
    return cer_score

FastVisionModel.for_inference(model)

print("── Baseline (before fine-tuning) ──")
baseline_val_cer = evaluate_cer(
    ds_val, val_references, model, tokenizer,
    n_samples=None, desc="Val baseline"
)
print(f"\nBaseline val CER: {baseline_val_cer:.4f}  (target: as low as possible)")

── Baseline (before fine-tuning) ──


Val baseline:   0%|          | 0/1 [00:00<?, ?batch/s]

  [Val baseline]  n=30  CER = 1.3273  (132.73 %)

Baseline val CER: 1.3273  (target: as low as possible)


In [17]:
# ── Clean up eval VRAM before training ──────────────────────────────────────
import gc
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
torch.cuda.synchronize()
# Delete any lingering eval tensors still in the notebook namespace
for _var in ['inputs', 'output_ids']:
    if _var in dir():
        del _var

gc.collect()                    # Python garbage collector
torch.cuda.empty_cache()        # Release cached-but-free blocks back to CUDA
# del trainer
print("VRAM after cleanup:")
print(f"  Allocated : {torch.cuda.memory_allocated()  / 1024**3:.2f} GB")
print(f"  Reserved  : {torch.cuda.memory_reserved()   / 1024**3:.2f} GB")

VRAM after cleanup:
  Allocated : 1.65 GB
  Reserved  : 1.69 GB


### 11 · Live Training Dashboard

In [18]:
_steps_per_epoch = max(1, len(converted_train) //
                       (PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION))
_total_steps = MAX_STEPS if MAX_STEPS > 0 else _steps_per_epoch * NUM_EPOCHS

EVAL_EVERY_N_STEPS = _steps_per_epoch   # eval exactly once per epoch

print(f"Steps / epoch : {_steps_per_epoch}")
print(f"Total steps   : {_total_steps}")
print(f"Eval every    : {EVAL_EVERY_N_STEPS} steps (= 1× per epoch)")

Steps / epoch : 22
Total steps   : 110
Eval every    : 22 steps (= 1× per epoch)


In [19]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display, clear_output
import time
from transformers import TrainerCallback


class LiveDashboardCallback(TrainerCallback):
    """
    6-panel training dashboard:
      ┌──────────────┬──────────────┬─────────────┐
      │  Train Loss  │   Val CER    │ Loss Hist.  │
      ├──────────────┼──────────────┼─────────────┤
      │  LR Schedule │  Val CER Δ   │  Live Stats │
      └──────────────┴──────────────┴─────────────┘
    Val CER is evaluated every EVAL_EVERY_N_STEPS on VAL_EVAL_SUBSET samples.
    Full val + test CER are computed at the very end (separate cells).
    """

    def __init__(self, total_steps, dataset_val, val_refs,
                 model_ref, tokenizer_ref, baseline_cer):
        self.total_steps   = total_steps
        self.ds_val        = dataset_val
        self.val_refs      = val_refs
        self.model_ref     = model_ref
        self.tokenizer_ref = tokenizer_ref
        self.baseline_cer  = baseline_cer

        # Training series
        self.steps   = []
        self.losses  = []
        self.lrs     = []
        # Val CER series  (step, cer)
        self.cer_steps = [0]
        self.cer_vals  = [baseline_cer]

        self.start_time = None
        self.fig = None

    # ── lifecycle ──────────────────────────────────────────────────────────
    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        print(f"🚀 Training started — {self.total_steps} steps total\n")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or 'loss' not in logs:
            return
        self.steps.append(state.global_step)
        self.losses.append(logs['loss'])
        self.lrs.append(logs.get('learning_rate', 0.0))

        # ── mid-training val CER ───────────────────────────────────────────
        step = state.global_step
        if step > 0 and step % EVAL_EVERY_N_STEPS == 0:
            # Switch to inference mode, eval, then back to training mode
            FastVisionModel.for_inference(self.model_ref)
            cer = evaluate_cer(
                self.ds_val, self.val_refs,
                self.model_ref, self.tokenizer_ref,
                n_samples = VAL_EVAL_SUBSET,
                desc = f"Val CER step {step}",
            )
            FastVisionModel.for_training(self.model_ref)
            self.cer_steps.append(step)
            self.cer_vals.append(cer)
                # ← ADD THIS BLOCK
            if USE_WANDB:
                import wandb
                wandb.log({"val_cer": cer}, step=step)

        self._render(state)

    def on_train_end(self, args, state, control, **kwargs):
        self._render(state, final=True)
        if self.fig:
            self.fig.savefig("training_dashboard.png", dpi=150, bbox_inches='tight')
            print("\n💾 Dashboard saved → training_dashboard.png")

    # ── EMA helper ─────────────────────────────────────────────────────────
    @staticmethod
    def _ema(values, alpha=0.15):
        if not values: return []
        out = [values[0]]
        for v in values[1:]:
            out.append(alpha * v + (1 - alpha) * out[-1])
        return out

    # ── rendering ──────────────────────────────────────────────────────────
    def _render(self, state, final=False):
        clear_output(wait=True)
        plt.close('all')

        fig = plt.figure(figsize=(16, 9))
        fig.patch.set_facecolor('#0d1117')
        gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

        ax_loss  = fig.add_subplot(gs[0, 0])
        ax_cer   = fig.add_subplot(gs[0, 1])
        ax_hist  = fig.add_subplot(gs[0, 2])
        ax_lr    = fig.add_subplot(gs[1, 0])
        ax_delta = fig.add_subplot(gs[1, 1])
        ax_stats = fig.add_subplot(gs[1, 2])

        BG, FG   = '#161b22', '#e6edf3'
        C_LOSS   = '#ff7b72'
        C_EMA    = '#ffa657'
        C_LR     = '#79c0ff'
        C_CER    = '#3fb950'
        C_CERDLT = '#d2a8ff'

        for ax in [ax_loss, ax_cer, ax_hist, ax_lr, ax_delta, ax_stats]:
            ax.set_facecolor(BG)
            for sp in ax.spines.values(): sp.set_edgecolor('#30363d')
            ax.tick_params(colors=FG, labelsize=8)
            for lbl in [ax.xaxis.label, ax.yaxis.label, ax.title]:
                lbl.set_color(FG)

        # ── panel 1: train loss ────────────────────────────────────────────
        if self.losses:
            ax_loss.plot(self.steps, self.losses,
                         color=C_LOSS, alpha=0.3, lw=1, label='Raw')
            ax_loss.plot(self.steps, self._ema(self.losses),
                         color=C_EMA,  lw=2.2, label='EMA')
            ax_loss.legend(facecolor=BG, labelcolor=FG, fontsize=8)
        ax_loss.set_title('Training Loss', fontweight='bold')
        ax_loss.set_xlabel('Step'); ax_loss.set_ylabel('Loss')
        ax_loss.grid(True, alpha=0.12)

        # ── panel 2: val CER ──────────────────────────────────────────────
        ax_cer.plot(self.cer_steps, self.cer_vals,
                    color=C_CER, lw=2.2, marker='o', ms=5, label='Val CER')
        ax_cer.axhline(self.baseline_cer, color='#8b949e',
                       lw=1, ls='--', label='Baseline')
        ax_cer.set_title('Validation CER  (↓ better)', fontweight='bold')
        ax_cer.set_xlabel('Step'); ax_cer.set_ylabel('CER')
        ax_cer.legend(facecolor=BG, labelcolor=FG, fontsize=8)
        ax_cer.grid(True, alpha=0.12)
        if self.cer_vals:
            ax_cer.set_ylim(bottom=0,
                            top=max(self.baseline_cer * 1.15, max(self.cer_vals) * 1.1))

        # ── panel 3: loss histogram ────────────────────────────────────────
        if len(self.losses) > 4:
            ax_hist.hist(self.losses,
                         bins=min(30, max(5, len(self.losses)//3)),
                         color='#388bfd', alpha=0.8, edgecolor=BG)
        ax_hist.set_title('Loss Distribution', fontweight='bold')
        ax_hist.set_xlabel('Loss'); ax_hist.set_ylabel('Count')
        ax_hist.grid(True, alpha=0.12)

        # ── panel 4: LR schedule ───────────────────────────────────────────
        if self.lrs:
            ax_lr.plot(self.steps, self.lrs, color=C_LR, lw=2)
        ax_lr.set_title('Learning Rate', fontweight='bold')
        ax_lr.set_xlabel('Step'); ax_lr.set_ylabel('LR')
        ax_lr.grid(True, alpha=0.12)

        # ── panel 5: val CER Δ (improvement per eval) ─────────────────────
        if len(self.cer_vals) > 1:
            deltas = [self.cer_vals[0] - v for v in self.cer_vals]
            colors = [C_CER if d >= 0 else C_LOSS for d in deltas]
            ax_delta.bar(self.cer_steps, deltas, color=colors, alpha=0.8)
        ax_delta.axhline(0, color='#8b949e', lw=1)
        ax_delta.set_title('CER Improvement vs Baseline  (↑ better)', fontweight='bold')
        ax_delta.set_xlabel('Step'); ax_delta.set_ylabel('ΔCER')
        ax_delta.grid(True, alpha=0.12)

        # ── panel 6: live stats ────────────────────────────────────────────
        ax_stats.axis('off')
        elapsed  = time.time() - self.start_time
        step     = state.global_step
        pct      = step / max(self.total_steps, 1) * 100
        eta_str  = "—"
        if step > 0:
            eta_sec = elapsed / step * (self.total_steps - step)
            eta_str = f"{int(eta_sec//60)}m {int(eta_sec%60)}s"
        c_loss = self.losses[-1] if self.losses else 0
        b_loss = min(self.losses)  if self.losses else 0
        c_cer  = self.cer_vals[-1] if self.cer_vals else self.baseline_cer
        b_cer  = min(self.cer_vals) if self.cer_vals else self.baseline_cer
        label  = "✅ DONE" if final else "🔥 TRAINING"

        stats = (
            f"  {label}\n\n"
            f"  Step     {step:>6} / {self.total_steps}\n"
            f"  Progress {pct:>5.1f} %\n"
            f"  Elapsed  {int(elapsed//60)}m {int(elapsed%60)}s\n"
            f"  ETA      {eta_str}\n\n"
            f"  Loss (cur)  {c_loss:.4f}\n"
            f"  Loss (best) {b_loss:.4f}\n"
            f"  LR          {self.lrs[-1] if self.lrs else 0:.2e}\n\n"
            f"  Val CER (cur)   {c_cer:.4f}\n"
            f"  Val CER (best)  {b_cer:.4f}\n"
            f"  Baseline CER    {self.baseline_cer:.4f}\n"
        )
        ax_stats.text(0.05, 0.97, stats, transform=ax_stats.transAxes,
                      fontsize=9.5, va='top', fontfamily='monospace', color=FG,
                      bbox=dict(boxstyle='round,pad=0.7', facecolor='#0d419d',
                                alpha=0.9, edgecolor=C_LR, lw=1.5))
        ax_stats.set_title('Live Stats', fontweight='bold', color=FG)

        fig.suptitle('🏛️ Spanish OCR LoRA — Training Dashboard',
                     fontsize=14, fontweight='bold', color=FG, y=1.01)
        plt.show()
        self.fig = fig

print("✅ LiveDashboardCallback ready")

✅ LiveDashboardCallback ready


### 12 · Train

In [20]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

dashboard_cb = LiveDashboardCallback(
    total_steps   = _total_steps,
    dataset_val   = ds_val,
    val_refs      = val_references,
    model_ref     = model,
    tokenizer_ref = tokenizer,
    baseline_cer  = baseline_val_cer,
)

if USE_WANDB:
    import wandb
    wandb.init(project=WANDB_PROJECT, config={
        "model": MODEL_NAME, "lora_r": LORA_R, "lr": LEARNING_RATE,
        "epochs": NUM_EPOCHS, "batch_eff": PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION,
        "dropout": LORA_DROPOUT, "augmentation": USE_AUGMENTATION,
    })

model.config.image_size = IMAGE_MAX_DIM
# model.config.vision_config.image_size = IMAGE_MAX_DIM

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = converted_train,
    callbacks     = [dashboard_cb],
    args = SFTConfig(
        per_device_train_batch_size = PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps = GRADIENT_ACCUMULATION,
        num_train_epochs  = NUM_EPOCHS if MAX_STEPS <= 0 else 1,
        max_steps         = MAX_STEPS,

        learning_rate     = LEARNING_RATE,
        lr_scheduler_type = LR_SCHEDULER,
        warmup_steps      = WARMUP_STEPS,
        weight_decay      = WEIGHT_DECAY,
        max_grad_norm     = MAX_GRAD_NORM,
        optim             = "adamw_8bit",
        logging_steps     = LOGGING_STEPS,
        save_steps        = SAVE_STEPS,
        save_total_limit  = 3,
        output_dir        = OUTPUT_DIR,
        seed              = 3407,
        report_to         = "wandb" if USE_WANDB else "none",

        # Required for vision fine-tuning
        remove_unused_columns = False,
        dataset_text_field    = "",
        dataset_kwargs        = {"skip_prepare_dataset": True},
        # max_seq_length = MAX_SEQ_LENGTH, 
    ),
)

wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Model does not have a default image size - using 512


In [21]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory = round(gpu_stats.total_memory / 1024**3, 3)
print(f"GPU: {gpu_stats.name}  |  Total VRAM: {max_memory} GB")
print(f"Reserved before training: {start_gpu_memory} GB")

GPU: NVIDIA L4  |  Total VRAM: 22.034 GB
Reserved before training: 13.549 GB


In [23]:
trainer_stats = trainer.train()#resume_from_checkpoint=True)

/tmp/ipykernel_2500/2935606820.py:78: UserWarning: Glyph 9989 (\N{WHITE HEAVY CHECK MARK}) missing from current font.
  self.fig.savefig("training_dashboard.png", dpi=150, bbox_inches='tight')
/tmp/ipykernel_2500/2935606820.py:78: UserWarning: Glyph 127963 (\N{CLASSICAL BUILDING}) missing from current font.
  self.fig.savefig("training_dashboard.png", dpi=150, bbox_inches='tight')



💾 Dashboard saved → training_dashboard.png


In [24]:
used_memory  = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
lora_memory  = round(used_memory - start_gpu_memory, 3)
runtime_min  = round(trainer_stats.metrics['train_runtime'] / 60, 2)
print(f"\n{'═'*55}")
print(f"  Training complete in {runtime_min} minutes")
print(f"  Peak VRAM          : {used_memory} GB  ({round(used_memory/max_memory*100,1)} %)")
print(f"  LoRA VRAM overhead : {lora_memory} GB  ({round(lora_memory/max_memory*100,1)} %)")
print(f"{'═'*55}")
if USE_WANDB:
    wandb.finish()


═══════════════════════════════════════════════════════
  Training complete in 22.65 minutes
  Peak VRAM          : 13.549 GB  (61.5 %)
  LoRA VRAM overhead : 0.0 GB  (0.0 %)
═══════════════════════════════════════════════════════


train/epoch,▁▁▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇███
train/global_step,▁▁▂▂▂▃▃▃▄▄▄▄▅▅▅▆▆▆▇▇▇███
train/grad_norm,▃▂▂▂▂▁▂▁▁▁▁▁▂▁▂▂▁▁▁▂▁▁█
train/learning_rate,▂▃▄▅▆▇███▇▇▆▆▅▄▄▃▃▂▂▁▁▁
train/loss,██▇▆▅▄▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_cer,▁
total_flos,1.100884551708672e+16
train/epoch,5
train/global_step,115
train/grad_norm,13.31712
train/learning_rate,0.0


In [ ]:
# import shutil

# folder_path = "./outputs"  # change this to your folder
# zip_path = "./outputs.zip"

# shutil.make_archive(zip_path.replace(".zip", ""), 'zip', folder_path)

# print("✅ Zipped at:", zip_path)


### 13 · Full Validation CER (post-training)

In [25]:
from unsloth import FastVisionModel
import torch

# Load directly from the saved checkpoint — bypasses the entire
CHECKPOINT_PATH = "./outputs/checkpoint-440"   # adjust if numbered differently

model, tokenizer = FastVisionModel.from_pretrained(
    CHECKPOINT_PATH,
    load_in_4bit               = LOAD_IN_4BIT,
    use_gradient_checkpointing = "unsloth",
    attn_implementation        = "flash_attention_2",
)
tokenizer.padding_side = "left"
tokenizer.image_processor.padding_side = "left"
if hasattr(tokenizer, 'tokenizer'):
    tokenizer.tokenizer.padding_side = "left"

tokenizer.image_processor.max_pixels = IMAGE_MAX_DIM * IMAGE_MAX_DIM
tokenizer.image_processor.min_pixels = 256 * 256

print(f"✅ Loaded trained adapter from {CHECKPOINT_PATH}")
print(f"   dtype={model.dtype}")

RuntimeError: Unsloth: No config file found - are you sure the `model_name` is correct?
If you're using a model on your local device, confirm if the folder location exists.
If you're using a HuggingFace online model, check if it exists.

In [29]:
FastVisionModel.for_inference(model)
print("Running full validation set evaluation …")
final_val_cer = evaluate_cer(
    ds_val, val_references, model, tokenizer,
    n_samples = None,          # all 500 val samples
    desc      = "Full Val",
)
print(f"\n  Baseline Val CER : {baseline_val_cer:.4f}")
print(f"  Final    Val CER : {final_val_cer:.4f}")
print(f"  Improvement      : {(baseline_val_cer - final_val_cer):.4f}  "
      f"({(baseline_val_cer - final_val_cer) / baseline_val_cer * 100:.1f} %)")

Running full validation set evaluation …


Full Val:   0%|          | 0/1 [00:00<?, ?batch/s]

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


  [Full Val]  n=30  CER = 0.3338  (33.38 %)

  Baseline Val CER : 1.3273
  Final    Val CER : 0.3338
  Improvement      : 0.9935  (74.8 %)


In [26]:
# ── Full output display for debugging ────────────────────────────────────
import textwrap

def display_full_predictions(dataset_split, references, model, tokenizer,
                              n_samples=10, seed=42):
    """Show complete GT vs PRED side by side with per-sample CER."""
    from jiwer import cer as _cer
    import random
    random.seed(seed)
    indices = random.sample(range(len(dataset_split)), min(n_samples, len(dataset_split)))

    for rank, idx in enumerate(indices):
        img  = resize_image(dataset_split[idx]["image"])
        messages = [{"role": "user", "content": [
            {"type": "text", "text": INSTRUCTION},
            {"type": "image"},
        ]}]
        input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
        inputs = tokenizer(img, input_text, add_special_tokens=False,
                           return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens   = EVAL_MAX_NEW_TOKENS,
                use_cache        = True,
                do_sample        = False,
                # repetition_penalty = 1.3,    # ← penalises repeating the same token
                # no_repeat_ngram_size = 4,    # ← hard-blocks any 4-gram from repeating exactly
            )
        prompt_len = inputs["input_ids"].shape[1]
        pred = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)
        pred = truncate_repetition_loop(pred)
        gt   = references[idx]
        sample_cer = _cer(gt, pred)

        sep = "─" * 70
        print(f"\n{sep}")
        print(f"Sample {rank+1}/{n_samples}  |  dataset idx={idx}  |  CER={sample_cer:.4f}")
        print(f"{sep}")
        print("── GROUND TRUTH ──")
        print(gt)          # ← full, untruncated
        print("── PREDICTION ──")
        print(pred)        # ← full, untruncated
        print(f"{sep}\n")

# Run it
FastVisionModel.for_inference(model)
display_full_predictions(ds_val, val_references, model, tokenizer,
                         n_samples=1, seed=42)


──────────────────────────────────────────────────────────────────────
Sample 1/1  |  dataset idx=20  |  CER=0.1435
──────────────────────────────────────────────────────────────────────
── GROUND TRUTH ──
Debitorio

En tres Julio mil Setecientos sesenta y nueve, se
ha presentado en el Oficio de Hipotecas de esta ciudad de Gerona, una
escritura otorgada, ante Pablo Calsa Notario de la Villa de Cas-sa de la Selva, en veinte, y nueve Junio de dicho año, con la
qual Jayme Bosch Negociante de la Villa de Tossa, con-fesso dever a Don Francisco Rates Visitador de las rentas
reales, en la ciudad de Gerona domiciliado, la cantidad de
ciento, y treinta libras, y diez, y nueve sueldos /respectivamente
barceloneses; Y son por las causas en dicha Escritura contenidas;
Y promettio pagarlas, esto es con dos iguales plassos, a
saber el primero por todo el dia quinze Agosto, y el
otro por el dia de Navidad, todo respectivamente e inmediato venturo
A cuio cumplimiento obligo en su Persona, y todos sus

### 15 · Save Adapter & Push to HuggingFace Hub

In [27]:
import os
LOCAL_ADAPTER_DIR = "spanish_ocr_lora_adapter"

model.save_pretrained(LOCAL_ADAPTER_DIR)
tokenizer.save_pretrained(LOCAL_ADAPTER_DIR)
print(f"✅ Adapter saved locally → {LOCAL_ADAPTER_DIR}/")

✅ Adapter saved locally → spanish_ocr_lora_adapter/


In [31]:
# ── Write model card ──────────────────────────────────────────────────────
_test_cer_str = f"{test_cer:.4f}" if '_RUN_TEST_EVAL' in dir() and _RUN_TEST_EVAL else "N/A"

model_card = f"""---
language:
  - es
tags:
  - lora
  - vision
  - ocr
  - historical-documents
  - handwritten-text-recognition
  - unsloth
base_model: {MODEL_NAME}
datasets:
  - {HF_DATASET_NAME}
---

# 🏛️ Historical Spanish OCR LoRA

LoRA adapter for **{MODEL_NAME}** fine-tuned on historical Spanish manuscript images.

## Results

| Split | CER |
|---|---|
| Validation (baseline) | {baseline_val_cer:.4f} |
| Validation (fine-tuned) | {final_val_cer:.4f} |
| **Test (fine-tuned)** | **{_test_cer_str}** |

## Training configuration

| Param | Value |
|---|---|
| Base model | `{MODEL_NAME}` |
| LoRA r / α / dropout | {LORA_R} / {LORA_ALPHA} / {LORA_DROPOUT} |
| Learning rate | {LEARNING_RATE} ({LR_SCHEDULER} schedule) |
| Epochs | {NUM_EPOCHS} |
| Effective batch size | {PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION} |
| Max image dim | {IMAGE_MAX_DIM} px |
| Data augmentation | {USE_AUGMENTATION} |
| Train / Val / Test | ~2000 / 500 / 500 samples |

## Text normalisation applied

1. Long-s `ſ` → `s`
2. `ç`/`Ç` → `z`/`Z`
3. Nasal tilde abbreviations: `ã`→`an`, `õ`→`on`, `ẽ`→`en`, `ũ`→`un`, `ĩ`→`in`
4. D-with-stroke `đ` → `de`
5. Stress accents stripped (except `ñ`)
6. Multiple spaces collapsed

## Inference

```python
from unsloth import FastVisionModel
from PIL import Image

model, tokenizer = FastVisionModel.from_pretrained("{ADAPTER_REPO_ID}", load_in_4bit=False)
FastVisionModel.for_inference(model)

image = Image.open("manuscript.jpg")
w, h = image.size
if max(w, h) > {IMAGE_MAX_DIM}:
    scale = {IMAGE_MAX_DIM} / max(w, h)
    image = image.resize((int(w*scale), int(h*scale)))

messages = [{{"role": "user", "content": [
    {{"type": "text", "text": "Transcribe the text in this historical Spanish manuscript image."}},
    {{"type": "image"}}
]}}]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
```
"""

with open(os.path.join(LOCAL_ADAPTER_DIR, "README.md"), "w") as f:
    f.write(model_card)
print("✅ Model card written")

✅ Model card written


In [32]:
if HF_TOKEN:
    print(f"Pushing to https://huggingface.co/{ADAPTER_REPO_ID} …")
    model.push_to_hub(ADAPTER_REPO_ID, token=HF_TOKEN, private=True)
    tokenizer.push_to_hub(ADAPTER_REPO_ID, token=HF_TOKEN, private=True)

    from huggingface_hub import HfApi
    api = HfApi()
    for fname in ["training_dashboard.png", "eval_results.txt"]:
        if os.path.exists(fname):
            api.upload_file(
                path_or_fileobj = fname,
                path_in_repo    = fname,
                repo_id         = ADAPTER_REPO_ID,
                token           = HF_TOKEN,
            )
            print(f"  ↑ {fname}")
    print(f"\n✅ Done → https://huggingface.co/{ADAPTER_REPO_ID}")
else:
    print("⚠️  HF_TOKEN not set — skipping Hub push.")

Pushing to https://huggingface.co/Ak137/qwen3.5-0.8B-handwritten-spanish-ocr-lora …
Saved model to https://huggingface.co/Ak137/qwen3.5-0.8B-handwritten-spanish-ocr-lora
  ↑ training_dashboard.png
  ↑ eval_results.txt

✅ Done → https://huggingface.co/Ak137/qwen3.5-0.8B-handwritten-spanish-ocr-lora
